# Bronze para Silver

Limpeza básica e gravação das quatro tabelas Silver.

In [0]:
from pyspark.sql import functions as F

partidas_bronze = spark.table("workspace.bronze.campeonato_brasileiro_full")
gols_bronze = spark.table("workspace.bronze.campeonato_brasileiro_gols")
cartoes_bronze = spark.table("workspace.bronze.campeonato_brasileiro_cartoes")
estatisticas_bronze = spark.table("workspace.bronze.campeonato_brasileiro_estatisticas_full")

## Partidas

In [0]:
partidas = partidas_bronze.select(
    F.col("ID").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.to_date("data", "dd/MM/yyyy").alias("data_partida"),
    F.trim("hora").alias("hora"),
    F.trim("mandante").alias("mandante"),
    F.trim("visitante").alias("visitante"),
    F.trim("formacao_mandante").alias("formacao_mandante"),
    F.trim("formacao_visitante").alias("formacao_visitante"),
    F.trim("tecnico_mandante").alias("tecnico_mandante"),
    F.trim("tecnico_visitante").alias("tecnico_visitante"),
    F.trim("vencedor").alias("vencedor"),
    F.trim(F.regexp_replace("arena", "\u00a0", " ")).alias("arena"),
    F.col("mandante_Placar").cast("int").alias("placar_mandante"),
    F.col("visitante_Placar").cast("int").alias("placar_visitante"),
    F.upper(F.trim("mandante_Estado")).alias("estado_mandante"),
    F.upper(F.trim("visitante_Estado")).alias("estado_visitante"),
    F.expr("try_cast(arrecadacao as double)").alias("arrecadacao")
).withColumn(
    "resultado",
    F.when(F.col("placar_mandante") > F.col("placar_visitante"), "mandante")
     .when(F.col("placar_mandante") < F.col("placar_visitante"), "visitante")
     .otherwise("empate")
).withColumn("versao_fonte", F.lit(18))  .withColumn("processado_em", F.current_timestamp())

display(partidas.limit(10))

partida_id,rodada,data_partida,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,placar_mandante,placar_visitante,estado_mandante,estado_visitante,arrecadacao,resultado,versao_fonte,processado_em
1,1,2003-03-29,16:00:00,Guarani,Vasco,null,null,null,null,Guarani,Brinco de Ouro,4,2,SP,RJ,null,mandante,18,2026-09-22T22:39:33.522Z
2,1,2003-03-29,16:00:00,Athletico-PR,Gremio,null,null,null,null,Athletico-PR,Arena da Baixada,2,0,PR,RS,null,mandante,18,2026-09-22T22:39:33.522Z
3,1,2003-03-30,16:00:00,Flamengo,Coritiba,null,null,null,null,-,Maracanã,1,1,RJ,PR,null,empate,18,2026-09-22T22:39:33.522Z
4,1,2003-03-30,16:00:00,Goias,Paysandu,null,null,null,null,-,Serra Dourada,2,2,GO,PA,null,empate,18,2026-09-22T22:39:33.522Z
5,1,2003-03-30,16:00:00,Internacional,Ponte Preta,null,null,null,null,-,Beira Rio,1,1,RS,SP,null,empate,18,2026-09-22T22:39:33.522Z
6,1,2003-03-30,16:00:00,Criciuma,Fluminense,null,null,null,null,Criciuma,Heriberto Hulse,2,0,SC,RJ,null,mandante,18,2026-09-22T22:39:33.522Z
7,1,2003-03-30,16:00:00,Juventude,Sao Paulo,null,null,null,null,-,Alfredo Jaconi,2,2,RS,SP,null,empate,18,2026-09-22T22:39:33.522Z
8,1,2003-03-30,16:00:00,Fortaleza,Bahia,null,null,null,null,-,Castelão,0,0,CE,BA,null,empate,18,2026-09-22T22:39:33.522Z
9,1,2003-03-30,16:00:00,Cruzeiro,Sao Caetano,null,null,null,null,-,Mineirão,2,2,MG,SP,null,empate,18,2026-09-22T22:39:33.522Z
10,1,2003-03-30,16:00:00,Vitoria,Figueirense,null,null,null,null,-,Barradão,1,1,BA,SC,null,empate,18,2026-09-22T22:39:33.522Z


## Gols e cartões

In [0]:
def adiciona_minutos(df):
    return (
        df.withColumn("minuto_original", F.trim("minuto"))
          .withColumn("minuto_base", F.expr("try_cast(regexp_extract(minuto_original, '^([0-9]+)', 1) as int)"))
          .withColumn("minuto_acrescimo", F.expr("try_cast(regexp_extract(minuto_original, '[+]([0-9]+)', 1) as int)"))
          .withColumn("minuto_ordem", F.col("minuto_base") + F.coalesce("minuto_acrescimo", F.lit(0)))
          .drop("minuto")
    )

gols = gols_bronze.select(
    F.col("partida_id").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.trim("clube").alias("clube"),
    F.trim("atleta").alias("atleta"),
    F.col("minuto"),
    F.coalesce(F.trim("tipo_de_gol"), F.lit("Normal")).alias("tipo_gol")
)
gols = adiciona_minutos(gols).withColumn("versao_fonte", F.lit(18))     .withColumn("processado_em", F.current_timestamp())

cartoes = cartoes_bronze.select(
    F.col("partida_id").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.trim("clube").alias("clube"),
    F.trim("cartao").alias("cartao"),
    F.trim("atleta").alias("atleta"),
    F.expr("try_cast(num_camisa as int)").alias("numero_camisa"),
    F.trim("posicao").alias("posicao"),
    F.col("minuto")
)
cartoes = adiciona_minutos(cartoes).withColumn("versao_fonte", F.lit(18))     .withColumn("processado_em", F.current_timestamp())

## Estatísticas

In [0]:
estatisticas = estatisticas_bronze.select(
    F.col("partida_id").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.trim("clube").alias("clube"),
    F.col("chutes").cast("int").alias("chutes"),
    F.col("chutes_no_alvo").cast("int").alias("chutes_no_alvo"),
    F.expr("try_cast(regexp_replace(posse_de_bola, '%', '') as double)").alias("posse_bola_pct"),
    F.col("passes").cast("int").alias("passes"),
    F.expr("try_cast(regexp_replace(precisao_passes, '%', '') as double)").alias("precisao_passes_pct"),
    F.col("faltas").cast("int").alias("faltas"),
    F.col("cartao_amarelo").cast("int").alias("cartoes_amarelos"),
    F.col("cartao_vermelho").cast("int").alias("cartoes_vermelhos"),
    F.col("impedimentos").cast("int").alias("impedimentos"),
    F.col("escanteios").cast("int").alias("escanteios")
).withColumn(
    "estatisticas_disponiveis",
    F.col("posse_bola_pct").isNotNull()
).withColumn("versao_fonte", F.lit(18))  .withColumn("processado_em", F.current_timestamp())

display(estatisticas.filter("estatisticas_disponiveis").limit(10))

partida_id,rodada,clube,chutes,chutes_no_alvo,posse_bola_pct,passes,precisao_passes_pct,faltas,cartoes_amarelos,cartoes_vermelhos,impedimentos,escanteios,estatisticas_disponiveis,versao_fonte,processado_em
4741,13,Chapecoense,15,0,35.0,280,null,19,4,0,0,8,true,18,2026-09-22T22:39:44.934Z
4741,13,Flamengo,9,0,65.0,509,null,13,1,0,1,5,true,18,2026-09-22T22:39:44.934Z
4808,20,Palmeiras,10,0,44.0,384,null,18,4,0,1,5,true,18,2026-09-22T22:39:44.934Z
4808,20,Criciuma,10,0,56.0,471,null,17,3,0,1,6,true,18,2026-09-22T22:39:44.934Z
4833,22,Sao Paulo,5,0,57.0,481,null,10,1,0,3,2,true,18,2026-09-22T22:39:44.934Z
4833,22,Coritiba,9,0,43.0,360,null,14,5,0,1,11,true,18,2026-09-22T22:39:44.934Z
4846,23,Chapecoense,14,0,58.0,477,null,14,4,0,2,4,true,18,2026-09-22T22:39:44.934Z
4846,23,Gremio,15,0,42.0,357,null,20,2,0,1,9,true,18,2026-09-22T22:39:44.934Z
4853,24,Gremio,15,0,52.0,383,null,19,0,0,4,6,true,18,2026-09-22T22:39:44.934Z
4853,24,Fluminense,16,0,48.0,363,null,17,0,0,1,6,true,18,2026-09-22T22:39:44.934Z


## Validações

In [0]:
duplicadas = partidas.groupBy("partida_id").count().filter("count > 1").count()
gols_orfaos = gols.join(partidas.select("partida_id"), "partida_id", "left_anti").count()
cartoes_orfaos = cartoes.join(partidas.select("partida_id"), "partida_id", "left_anti").count()
estatisticas_orfas = estatisticas.join(partidas.select("partida_id"), "partida_id", "left_anti").count()

validacao = spark.createDataFrame([
    ("IDs duplicados em partidas", duplicadas),
    ("Gols sem partida", gols_orfaos),
    ("Cartões sem partida", cartoes_orfaos),
    ("Estatísticas sem partida", estatisticas_orfas),
], ["teste", "ocorrencias"])

display(validacao)

if validacao.filter("ocorrencias > 0").count() > 0:
    raise ValueError("Falha nas validações")

teste,ocorrencias
IDs duplicados em partidas,0
Gols sem partida,0
Cartões sem partida,0
Estatísticas sem partida,0


## Gravação

In [0]:
tabelas = {
    "partidas": partidas,
    "gols": gols,
    "cartoes": cartoes,
    "estatisticas": estatisticas,
}

for nome, df in tabelas.items():
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.silver.{nome}"))

In [0]:
contagens = [(nome, df.count()) for nome, df in tabelas.items()]
display(spark.createDataFrame(contagens, ["tabela", "registros"]))
spark.sql("SHOW TABLES IN workspace.silver").show(truncate=False)

tabela,registros
partidas,9165
gols,10820
cartoes,20953
estatisticas,18330


+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|silver  |cartoes     |false      |
|silver  |estatisticas|false      |
|silver  |gols        |false      |
|silver  |partidas    |false      |
+--------+------------+-----------+

